# Ensemble Methods

Ensemble methods combine multiple models to produce stronger predictions than any single model alone.  
Three key strategies are explored here:

1. **Hard Voting** — majority vote from diverse classifiers  
2. **Bagging** — bootstrap-aggregated decision trees (variance reduction)  
3. **Random Forest** — bagging with random feature subsets (additional diversity)

**Dataset:** Wine Recognition — classify wine cultivar from 13 chemical measurements.

In [ ]:
import sys
sys.path.insert(0, r"/Jana CMOR/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning")
from rice_ml.supervised_learning.ensemble import hard_voting_classifier, bagging_classifier, random_forest_classifier
from rice_ml.supervised_learning.decision_tree_classifier import decision_tree_classifier
from rice_ml.supervised_learning.knn import KNN
from rice_ml.supervised_learning.logistic_regression import LogisticRegression
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Imports complete")

## Load & Explore the Dataset

The **Wine dataset** contains 178 samples from 3 Italian wine cultivars with 13 numeric chemical features (alcohol, malic acid, ash, etc.).

In [ ]:
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(class_names)}")
print(f"Class distribution: {dict(zip(class_names, np.bincount(y)))}")

## Exploratory Data Analysis

Visualising feature distributions and class separability helps understand the dataset before fitting ensemble models.

In [ ]:
# Class distribution & key feature scatter
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ['#e63946', '#457b9d', '#2a9d8f']

# Pie chart
axes[0].pie(np.bincount(y), labels=class_names, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title("Class Balance", fontsize=12, fontweight='bold')

# Scatter: Alcohol vs Flavanoids
for cls, color, name in zip(range(3), colors, class_names):
    mask = y == cls
    axes[1].scatter(X[mask, 0], X[mask, 6], color=color, alpha=0.7,
                    label=name, edgecolors='k', linewidths=0.4, s=50)
axes[1].set_xlabel("Alcohol", fontsize=11)
axes[1].set_ylabel("Flavanoids", fontsize=11)
axes[1].set_title("Alcohol vs Flavanoids", fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.5)

# Feature variances (helpful features have high between-class variation)
feature_vars = np.var(X, axis=0)
sorted_idx = np.argsort(feature_vars)[::-1][:8]
axes[2].barh(range(8), feature_vars[sorted_idx], color='steelblue', edgecolor='k', alpha=0.8)
axes[2].set_yticks(range(8))
axes[2].set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=8)
axes[2].set_xlabel("Feature Variance", fontsize=11)
axes[2].set_title("Top 8 Feature Variances", fontsize=12, fontweight='bold')
axes[2].grid(True, axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Preprocessing & Split

Standardisation ensures that KNN and Logistic Regression (distance/gradient-based) are not dominated by high-variance features like Proline.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Single Decision Tree Baseline

We establish a baseline with a single decision tree to motivate the ensemble approach.

In [ ]:
base_tree = decision_tree_classifier(max_depth=4)
base_tree.fit(X_train, y_train)
base_acc = base_tree.score(X_test, y_test)
print(f"Single Decision Tree Accuracy (depth=4): {base_acc:.4f}")

## Hard Voting Classifier

Combines a Decision Tree, KNN, and Logistic Regression by majority vote. Each model votes independently and the most popular class wins. Diverse base learners reduce correlated errors.

In [ ]:
tree  = decision_tree_classifier(max_depth=4)
knn   = KNN(k=5)
lr    = LogisticRegression(learning_rate=0.1, n_iterations=300)

voter = hard_voting_classifier(classifiers=[tree, knn, lr])
voter.fit(X_train, y_train)
voter_acc = voter.score(X_test, y_test)

# Evaluate each component alone
component_accs = {}
for name, clf in [("Decision Tree", tree), ("KNN", knn), ("Logistic Regression", lr)]:
    component_accs[name] = np.mean(np.array(clf.predict(X_test)) == y_test)

component_accs["Hard Voting"] = voter_acc
print(f"Hard Voting Accuracy: {voter_acc:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = ['#a8dadc', '#a8dadc', '#a8dadc', '#e63946']
bars = ax.bar(list(component_accs.keys()), list(component_accs.values()),
              color=colors_bar, edgecolor='k', alpha=0.85)
for bar, acc in zip(bars, component_accs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.3f}", ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0.7, 1.05)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Hard Voting vs Individual Classifiers", fontsize=13, fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Bagging Classifier

Bootstrap Aggregating (**Bagging**) trains many decision trees on different random subsamples of training data. The variance of the ensemble is lower than any individual tree.

In [ ]:
results_bag = {}
for n_est in [5, 10, 25, 50]:
    bag = bagging_classifier(n_estimators=n_est, max_depth=6, random_state=42)
    bag.fit(X_train, y_train)
    results_bag[n_est] = bag.score(X_test, y_test)
    print(f"Bagging n={n_est:>3}: {results_bag[n_est]:.4f}")

## Random Forest Classifier

Random Forest adds **feature randomness** on top of bagging. Each split only considers a random subset of features, decorrelating trees and further reducing variance.

In [ ]:
results_rf = {}
for n_est in [5, 10, 25, 50]:
    rf = random_forest_classifier(n_estimators=n_est, max_depth=6,
                                  max_features='sqrt', random_state=42)
    rf.fit(X_train, y_train)
    results_rf[n_est] = rf.score(X_test, y_test)
    print(f"Random Forest n={n_est:>3}: {results_rf[n_est]:.4f}")

## Model Comparison — Visualisation

Comparing all models on a single plot makes the ensemble advantage immediately visible.

In [ ]:
best_bag = max(results_bag, key=results_bag.get)
best_rf  = max(results_rf,  key=results_rf.get)

models = {
    'Decision Tree': base_acc,
    'Hard Voting':   voter_acc,
    f'Best Bagging\n(n={best_bag})':  results_bag[best_bag],
    f'Best RF\n(n={best_rf})':         results_rf[best_rf],
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy comparison bar chart
bar_colors = ['#a8dadc', '#457b9d', '#2a9d8f', '#e63946']
bars = axes[0].bar(range(len(models)), list(models.values()), color=bar_colors, edgecolor='k', alpha=0.85)
for bar, (name, acc) in zip(bars, models.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f"{acc:.3f}", ha='center', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(models)))
axes[0].set_xticklabels(list(models.keys()), rotation=10, ha='right', fontsize=10)
axes[0].set_ylim(0.8, 1.05)
axes[0].set_ylabel("Test Accuracy", fontsize=12)
axes[0].set_title("Model Comparison", fontsize=13, fontweight='bold')
axes[0].axhline(base_acc, color='gray', linestyle='--', linewidth=1.5, label=f'Baseline: {base_acc:.3f}')
axes[0].legend(fontsize=9)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.5)

# n_estimators vs accuracy (bagging vs RF)
n_ests = sorted(results_bag.keys())
axes[1].plot(n_ests, [results_bag[n] for n in n_ests], 'o-',
             color='#457b9d', linewidth=2.5, markersize=8, label='Bagging')
axes[1].plot(n_ests, [results_rf[n] for n in n_ests], 's-',
             color='#e63946', linewidth=2.5, markersize=8, label='Random Forest')
axes[1].axhline(base_acc, color='gray', linestyle='--', linewidth=1.5, label=f'Single Tree: {base_acc:.3f}')
axes[1].set_xlabel("n_estimators", fontsize=12)
axes[1].set_ylabel("Test Accuracy", fontsize=12)
axes[1].set_title("n_estimators vs Accuracy", fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].set_xticks(n_ests)

plt.tight_layout()
plt.show()

## Key Takeaways

- **Hard Voting** leverages diverse model types to reduce individual weaknesses.
- **Bagging** reduces variance by training on different bootstrap samples of data.
- **Random Forest** adds feature randomness, further decorrelating individual trees.
- Ensemble accuracy consistently exceeds the single decision tree baseline.
- More estimators generally helps but with diminishing returns — pick the minimum for your accuracy target.